### Import libaries. 

In [43]:
import os 
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

from langchain_community.document_loaders import YoutubeLoader
from langchain_cohere import CohereEmbeddings, ChatCohere
from langchain_community.vectorstores import FAISS 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate

import tiktoken
from dotenv import load_dotenv

In [11]:
# import the api key for using the model. 
os.environ['COHERE_API_KEY'] = "cohere_5iQczItcuqnmN02zj7Tue4cEvswgbT6d1x8uQTwQ3M1gJx"

In [7]:
# install the required libraries. 
%pip install -q youtube-transcript-api langchain-community langchain-cohere faiss-cpu tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [13]:
%pip install -q langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [8]:
# install new version of pip 
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [9]:
# verification of installation. 
from youtube_transcript_api import YouTubeTranscriptApi

from langchain_community.document_loaders import YoutubeLoader
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

import tiktoken
from dotenv import load_dotenv

print("All libraries imported successfully!")

C:\Users\Mohd Uzaif\AppData\Local\Temp\ipykernel_23056\1454285622.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import YoutubeLoader


All libraries imported successfully!


## Indexing.

### Document Ingestion.

In [20]:
video_id = "xAt1xcC6qfM" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi().fetch(
        video_id, 
        languages=["en"]
    )

    # Flatten it to plain text
    transcript = " ".join(snippet.text for snippet in transcript_list)
    
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

what's the biggest misunderstanding about you whenever you hear about somebody who's got you know ridiculous amounts of money their values are different than your values you should be concerned about their agenda what's your biggest Fe I'll be sad as my brain gets less capable that disappoints me tell me one Behavior we all should adopt reading a lot being a student that's a big part of My Success richest and most powerful men in the world Bill Gates has Unleashed a technological Revolution that has changed our lives Bill Gates of 25 and Bill Gates of 17 any change that you feel personally in my 20s being a maniac was the right thing my competitors would say oh no you work too hard and I'd say yes I do if you're in a race your 20s when you have no wife and no children that's the time to do it if you get an opportunity to invite three Indians for dinner who would that be there was a mathematician ramanujan I would have loved to have met him why do you think India is becoming a Global Ta

In [24]:
# transcript_list

### Text Splitting.

In [25]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [26]:
print(len(chunks))

42


In [ ]:
chunks

### Embedding Generation and Storing in Vector Store

In [29]:
# create the embedding model. 
embedding_model = CohereEmbeddings(
    model = "embed-english-v3.0"
)

# create the vector store for embedding. 
vector_store = FAISS.from_documents(chunks, embedding_model)

In [31]:
vector_store.index_to_docstore_id

{0: 'c1baf8bd-8dd5-4762-979a-39d29e10c43b',
 1: '1aa31c15-518f-400c-bf51-407817264c58',
 2: '2b1006e5-288c-408e-8da1-57d93dfc9b65',
 3: '45b1a957-5c15-4071-b22a-7aad32b1f7f1',
 4: '59ce1d2e-db13-4245-bb51-bf0de128cd19',
 5: 'd6807c32-fa4a-4a39-b3f8-1c121680df7c',
 6: 'b19898f5-dcac-48a7-8be1-b336f25b9768',
 7: '9a1cda3c-db0c-4234-a856-84587de9621a',
 8: '7226750a-9cde-49e6-bab5-53eaf6dd739e',
 9: '422f7142-2c0e-45d6-ae0b-8f5acfc39000',
 10: '096c471c-519b-47d1-b71b-43330997c6a0',
 11: '3318b0e3-7fec-4459-8a70-8ad4aba2270b',
 12: '3884d36a-55c2-4e40-a345-df19db4acd40',
 13: '69b5ce2f-d5a7-449f-8e76-84f62c44026b',
 14: '68e4b457-00b8-4356-b4fa-8c4de0032ee8',
 15: 'ea349f35-b66d-47c0-83c6-b63a4d023991',
 16: '44b1f787-d7e4-4ab9-8522-365c42deca29',
 17: '878f7169-1675-4af6-affc-921c0d5af0a8',
 18: '0b81dea4-7d2c-429e-8364-ce96221fdb99',
 19: '7acd8339-7d34-42b8-8cfb-41c72beed170',
 20: '7c879a52-0196-4e23-836f-eca63ae21589',
 21: '3f632388-667e-4ba9-9942-f3465ce151fa',
 22: '6aefe078-f3ee-

In [33]:
vector_store.get_by_ids('9fb1761a-f630-428b-b79d-0e981b755965')

[]

## Retrival.

In [34]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [35]:
retriever

VectorStoreRetriever(tags=['FAISS', 'CohereEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024A09445160>, search_kwargs={'k': 4})

In [36]:
retriever.invoke('What is deepmind')

[Document(id='baa098c7-5173-42cc-90c1-c46d1f0fc073', metadata={}, page_content="on uh you know AI just staying on top of that is you know I I very much enjoy that and I get to sit and talk with the top people at open Ai and I get to play around with things but then when you think okay now about what about AI applied to Mental Health Care uh you know isn't that one of the most exciting things because you know we can never have enough therapists uh and even people who aren't uh you know suffering massively you know maybe we could help uh even people with mild uh symptoms so you know what does this AI companion look like and you know how can that help us uh my friend Reed Hoffman uh just wrote a book called Super agency that he's got a really good about this so I'm pushing myself to try and understand uh you know what can we do there and meet the people who are you know pushing the the boundaries uh because I see such potential so what would you advise young people to start learning today

## Augmentation.

In [37]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [46]:
question          = "is the topic Artificial Intelligence discussed? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [47]:
retrieved_docs

[Document(id='baa098c7-5173-42cc-90c1-c46d1f0fc073', metadata={}, page_content="on uh you know AI just staying on top of that is you know I I very much enjoy that and I get to sit and talk with the top people at open Ai and I get to play around with things but then when you think okay now about what about AI applied to Mental Health Care uh you know isn't that one of the most exciting things because you know we can never have enough therapists uh and even people who aren't uh you know suffering massively you know maybe we could help uh even people with mild uh symptoms so you know what does this AI companion look like and you know how can that help us uh my friend Reed Hoffman uh just wrote a book called Super agency that he's got a really good about this so I'm pushing myself to try and understand uh you know what can we do there and meet the people who are you know pushing the the boundaries uh because I see such potential so what would you advise young people to start learning today

In [48]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"on uh you know AI just staying on top of that is you know I I very much enjoy that and I get to sit and talk with the top people at open Ai and I get to play around with things but then when you think okay now about what about AI applied to Mental Health Care uh you know isn't that one of the most exciting things because you know we can never have enough therapists uh and even people who aren't uh you know suffering massively you know maybe we could help uh even people with mild uh symptoms so you know what does this AI companion look like and you know how can that help us uh my friend Reed Hoffman uh just wrote a book called Super agency that he's got a really good about this so I'm pushing myself to try and understand uh you know what can we do there and meet the people who are you know pushing the the boundaries uh because I see such potential so what would you advise young people to start learning today and from where AI itself well if you have a mathematical mind you know not\n\n

In [49]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [50]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      on uh you know AI just staying on top of that is you know I I very much enjoy that and I get to sit and talk with the top people at open Ai and I get to play around with things but then when you think okay now about what about AI applied to Mental Health Care uh you know isn't that one of the most exciting things because you know we can never have enough therapists uh and even people who aren't uh you know suffering massively you know maybe we could help uh even people with mild uh symptoms so you know what does this AI companion look like and you know how can that help us uh my friend Reed Hoffman uh just wrote a book called Super agency that he's got a really good about this so I'm pushing myself to try and understand uh you know what can we do there and meet the people who are you know pushing t

In [51]:
# initialize the chat model. 
chat_model = ChatCohere(
    model="command-a-03-2025",
    temperature=0
)

In [52]:
answer = chat_model.invoke(final_prompt)
print(answer.content)

Yes, the topic of Artificial Intelligence (AI) is discussed in the provided transcript. Here are the key points that were covered:

1. **AI in Mental Health Care**: The potential of AI to assist in mental health care is highlighted, particularly in addressing the shortage of therapists and helping individuals with mild symptoms. The speaker mentions exploring how an AI companion could be designed to support mental health.

2. **Learning and Opportunities in AI**: The speaker advises young people with a mathematical mind to start learning about AI, emphasizing its vast potential. They note that while big companies are investing heavily in AI, there are still opportunities for new companies to emerge and achieve significant success, though it may be more challenging compared to earlier days in computing.

3. **AI's Impact and Risks**: The speaker expresses hope that AI will be shaped in a positive way, given its potential to surpass human intelligence and significantly impact the world. 

## Building a Chain.

In [53]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [54]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [55]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [56]:
parallel_chain.invoke('who is Demis')

{'context': "it's an incredible opportunity I'm so grateful thank you so much for doing this it's a dream come true great we'll have to do it again yes we need to I've got you on camera now thank you thank you pleasure I'm definitely nervous it's a big big big time it's time let's go I'm excited hi hi Raj how are you very good how are you pleasure meeting you so yes that's fun it's it's a pleasure seeing you Beau I'm pretty nervous at this point so uh just bear with me it be good good I had 500,000 things to talk to you and now I'm speechless and like there's nothing in my thank you so much for watching this episode till the end please let us know in the comments who are the next guests that you want to see on this show because we are determined to get the best of the best Minds from the world and provide you the maximum value I'll see you next time until then keep figuring out and don't forget to share this episode with at least one person just life positive change [Music]\n\na great 

In [57]:
parser = StrOutputParser()

In [59]:
main_chain = parallel_chain | prompt | chat_model | parser

In [60]:
main_chain.invoke('Can you summarize the video')

"The video appears to be an episode of a podcast or interview series where the host is speaking with Bill Gates. The host expresses excitement and gratitude for the opportunity to interview Gates, whom they have long admired. They mention watching a Netflix documentary about Gates during the pandemic and jokingly predicting they would one day interview him, which has now come true. The conversation touches on Gates' frequent visits to India and his work with the Gates Foundation, with the host expressing a desire to learn from Gates' experiences and insights. The host also reflects on their childhood admiration for Gates as the richest person in the world and the surreal experience of now interviewing him. The episode ends with the host thanking Gates and expressing their excitement for future episodes, encouraging viewers to subscribe and suggest future guests. \n\n**Summary:** The video is an interview with Bill Gates, where the host shares their journey of admiration for Gates, disc